# 03 — All Five Agents
Interact with InformationAgent, KnowledgeAgent, MetadataAgent, CapacityAgent, and RuleAgent directly.

In [ ]:
import sys; sys.path.insert(0, '/home/claude/codebase/code/src')
import os; os.environ['ENABLE_MOCK']='true'; os.environ['REDIS_ENABLED']='false'

## 1. InformationAgent — SQL metrics from Databricks (mock)

In [ ]:
from agents.information_agent import InformationAgent
from core.base_agent import AgentRequest

agent = InformationAgent()
req = AgentRequest(query='What is the GRR for retention?', data_products=['retention'])
result = agent.execute(req)
print('Success:', result.success)
print('Message:', result.message)
print('Confidence:', result.confidence)
print('Metrics:', result.data['metrics'])
print('Anomalies:', result.data['anomalies'])

### Low GRR scenario — anomaly detection

In [ ]:
from services.databricks.mock import MockDatabricksService
low_grr_agent = InformationAgent(data_service=MockDatabricksService(low_grr=True))
result = low_grr_agent.execute(AgentRequest(query='Retention metrics', data_products=['retention']))
print('Anomalies detected:', result.data['anomalies'])

## 2. KnowledgeAgent — RAG over governance docs

In [ ]:
from agents.knowledge_agent import KnowledgeAgent

agent = KnowledgeAgent()
result = agent.execute(AgentRequest(query='What is gross retention rate?'))
print('Success:', result.success)
print('Docs found:', len(result.data['knowledge']))
for doc in result.data['knowledge']:
    print(f"  [{doc['topic']}] {doc['definition'][:80]}...")

## 3. MetadataAgent — Collibra asset metadata

In [ ]:
from agents.metadata_agent import MetadataAgent

agent = MetadataAgent()
result = agent.execute(AgentRequest(query='Who owns retention data?', data_products=['retention']))
print('Success:', result.success)
print('Message:', result.message)
for product, meta in result.data.items():
    print(f"\nProduct: {product}")
    print(f"  Asset: {meta['asset_name']} ({meta['asset_id']})")
    print(f"  Owner: {meta['owner']}")
    print(f"  DQ Score: {meta['data_quality'].get('score')}%")

## 4. CapacityAgent — Jira ticket management

In [ ]:
from agents.capacity_agent import CapacityAgent
from services.jira.mock import MockJiraService

svc = MockJiraService()
agent = CapacityAgent(ticket_service=svc)

# List open tickets
result = agent.execute(AgentRequest(query='Show open incidents', data_products=['retention']))
print('Open tickets:', result.metadata['open_issues'])
for t in result.data.get('retention', []):
    print(f"  {t['id']}: {t['summary']} [{t['status']}]")

# Create a ticket
result2 = agent.create_ticket_from_anomaly('GRR dropped to 78%', product='retention')
print('\nCreated:', result2.data['ticket_id'])
print('Tickets in service:', len(svc.tickets))

## 5. RuleAgent — Rule registry CRUD

In [ ]:
from agents.rule_agent import RuleAgent, RULE_REGISTRY

agent = RuleAgent()

# List all seed rules
result = agent.execute(AgentRequest(query='list all rules'))
print(f'Total rules: {len(result.data)}')
for r in result.data:
    print(f"  {r['id']}: {r['name']} ({r['type']})")

In [ ]:
# Create a new DQ rule
from core.base_agent import AgentRequest
result = agent.execute(AgentRequest(
    query='create rule for completeness check',
    data_products=['retention'],
    context={
        'rule_name': 'Retention NULL Check',
        'dimension': 'completeness',
        'asset': 'analytics.retention_metrics',
        'expression': 'null_pct < 0.01',
        'severity': 'High',
    }
))
print('Created rule:', result.data['id'], '-', result.data['name'])

In [ ]:
# Evaluate rules
result = agent.execute(AgentRequest(query='evaluate rules', data_products=['retention']))
print('Evaluation results:')
for r in result.data:
    status = '✅' if r['passed'] else '❌'
    print(f"  {status} {r['rule_id']}: {r['rule_name']}")
print('\nMetadata:', result.metadata)

## Agent health checks

In [ ]:
from agents.information_agent import InformationAgent
from agents.knowledge_agent import KnowledgeAgent
from agents.metadata_agent import MetadataAgent
from agents.capacity_agent import CapacityAgent
from agents.rule_agent import RuleAgent

for AgentClass in [InformationAgent, KnowledgeAgent, MetadataAgent, CapacityAgent, RuleAgent]:
    health = AgentClass().health_check()
    status = '🟢' if health.get('healthy') else '🔴'
    print(f"{status} {health['agent']}: {health}")